In [1]:
import xarray as xr
import numpy as np
import json
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from global_land_mask import globe 

def process_mwp_high_res_with_custom_cbar():
    print("GENERATOR SPATIAL PLOT: MEAN WAVE PERIOD (MWP)")

    # 0. KONFIGURASI TAMPILAN
    TEXT_COLOR = "#e0e0e0" 
    CMAP_NAME = "viridis"  # Tetap menggunakan tema yang sama agar konsisten
    DPI_OUTPUT = 200        

    # 1. KONFIGURASI DATA
    start_year = 1979
    end_year = 2024
    data_folder = '.' 
    
    # [MODIFIKASI 1] Ubah pola pencarian file ke Mean Wave Period
    file_pattern = "mean_wave_period" 
    
    # [MODIFIKASI 2] Ubah folder output agar tidak menimpa hasil SWH
    base_frame_dir = "frames_mwp"
    output_dirs = [f"{base_frame_dir}/ts", f"{base_frame_dir}/clim", f"{base_frame_dir}/seas"]
    for d in output_dirs:
        os.makedirs(d, exist_ok=True)

    monthly_datasets = []

    # 2. LOAD DATA
    print(f"Membaca data MWP {start_year}-{end_year}...")

    for year in range(start_year, end_year + 1):
        search_path = os.path.join(data_folder, f"*{file_pattern}*{year}*.nc")
        files = glob.glob(search_path)
        if not files: continue
        
        try:
            ds = xr.open_dataset(files[0])
            if 'valid_time' in ds.coords: ds = ds.rename({'valid_time': 'time'})
            
            # [MODIFIKASI 3] Deteksi nama variabel untuk MWP
            var_name = None
            possible_names = ['mwp', 'mean_wave_period', 'mean_period', 'mp']
            for v in possible_names:
                if v in ds:
                    var_name = v
                    break
            
            if var_name:
                ds_month = ds[var_name].resample(time='1M').mean()
                monthly_datasets.append(ds_month)
            ds.close()
            print(f"   [{year}] OK", end='\r')
        except Exception:
            pass

    if not monthly_datasets:
        print("\nGagal: Tidak ada data Mean Wave Period ditemukan.")
        return

    print("\nMenggabungkan dataset...")
    ds_combined = xr.concat(monthly_datasets, dim='time')
    ds_clim = ds_combined.groupby('time.month').mean()
    ds_seas = ds_combined.groupby('time.season').mean()

    # Hitung Range Warna
    ranges = {
        'ts': [float(ds_combined.min()), float(ds_combined.max())],
        'clim': [float(ds_clim.min()), float(ds_clim.max())],
        'seas': [float(ds_seas.min()), float(ds_seas.max())]
    }

    # ==========================================
    # 3. FUNGSI PEMBUAT COLORBAR (FIXED BEAUTIFUL NUMBERS)
    # ==========================================
    def generate_custom_colorbar(vmin, vmax, output_path):
        import matplotlib.ticker as ticker
        # Gunakan levels 200 agar gradasi warna viridis terlihat sangat halus
        levels = np.linspace(vmin, vmax, 200)
        fig_cbar = plt.figure(figsize=(1.5, 6), dpi=DPI_OUTPUT)
        fig_cbar.patch.set_alpha(0) 
        
        ax_dummy = fig_cbar.add_axes([0.1, 0.05, 0.3, 0.9])
        dummy_cntr = ax_dummy.contourf([[0,0],[0,0]], levels=levels, cmap=CMAP_NAME)
        ax_dummy.remove() 
        
        cbar_ax = fig_cbar.add_axes([0.2, 0.05, 0.4, 0.9])
        cbar = plt.colorbar(dummy_cntr, cax=cbar_ax, orientation="vertical")
        
        # FIX: Menggunakan Locator untuk angka periode gelombang (detik) yang bulat
        tick_locator = ticker.MaxNLocator(nbins=8, prune=None)
        cbar.locator = tick_locator
        cbar.update_ticks()
        
        # Tetap menggunakan label satuan Seconds (s) sesuai data MWP
        cbar.set_label("Mean Wave Period (s)", color=TEXT_COLOR, fontsize=14, fontweight='bold')
        
        cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
        plt.setp(cbar.ax.get_yticklabels(), color=TEXT_COLOR, fontsize=12, fontweight='bold')
        cbar.outline.set_edgecolor(TEXT_COLOR)
        cbar.ax.set_facecolor('none')
        
        plt.savefig(output_path, transparent=True, bbox_inches='tight')
        plt.close(fig_cbar)

    # ==========================================
    # 4. FUNGSI PLOT
    # ==========================================
    def generate_style_plot(data_slice, filename, vmin, vmax):
        lat_min = data_slice.latitude.min().values
        lat_max = data_slice.latitude.max().values
        lon_min = data_slice.longitude.min().values
        lon_max = data_slice.longitude.max().values
        
        smoothing_factor = 5
        n_lat = int(len(data_slice.latitude) * smoothing_factor)
        n_lon = int(len(data_slice.longitude) * smoothing_factor)

        new_lat = np.linspace(lat_min, lat_max, n_lat)
        new_lon = np.linspace(lon_min, lon_max, n_lon)
        data_smooth = data_slice.interp(latitude=new_lat, longitude=new_lon, method='linear')

        lon_grid, lat_grid = np.meshgrid(new_lon, new_lat)
        is_on_land = globe.is_land(lat_grid, lon_grid)
        data_values_masked = np.where(is_on_land, np.nan, data_smooth.values)

        aspect = (lon_max - lon_min) / (lat_max - lat_min)
        fig = plt.figure(figsize=(5 * aspect, 5), dpi=DPI_OUTPUT, frameon=False)
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
        
        ax.pcolormesh(
            new_lon, new_lat, data_values_masked,
            transform=ccrs.PlateCarree(),
            vmin=vmin, vmax=vmax,
            cmap=CMAP_NAME,
            shading='auto',
            zorder=0 
        )
        ax.add_feature(cfeature.LAND, facecolor='none', edgecolor='black', linewidth=1.5, zorder=10)
        ax.set_axis_off()
        try: ax.spines['geo'].set_visible(False)
        except: pass
        
        plt.savefig(filename, transparent=True, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

    # ==========================================
    # 5. EKSEKUSI GENERATE FRAME
    # ==========================================
    
    print("\nProcessing Time Series...")
    generate_custom_colorbar(ranges['ts'][0], ranges['ts'][1], f"{base_frame_dir}/ts/colorbar_legend.png")
    labels_ts = []
    for i, t in enumerate(ds_combined.time):
        fname = f"{base_frame_dir}/ts/spatial_ts_{i}.png"
        generate_style_plot(ds_combined.isel(time=i), fname, ranges['ts'][0], ranges['ts'][1])
        labels_ts.append(pd.to_datetime(t.values).strftime('%Y-%m'))
        if i%50==0: print(f"   {i} frames...", end='\r')

    print("\nProcessing Monthly Clim...")
    generate_custom_colorbar(ranges['clim'][0], ranges['clim'][1], f"{base_frame_dir}/clim/colorbar_legend.png")
    for i in range(12):
        fname = f"{base_frame_dir}/clim/spatial_clim_{i}.png"
        generate_style_plot(ds_clim.sel(month=i+1), fname, ranges['clim'][0], ranges['clim'][1])

    print("\nProcessing Seasonal...")
    generate_custom_colorbar(ranges['seas'][0], ranges['seas'][1], f"{base_frame_dir}/seas/colorbar_legend.png")
    season_order = ['DJF', 'MAM', 'JJA', 'SON']
    for i, s in enumerate(season_order):
        try:
            fname = f"{base_frame_dir}/seas/spatial_seas_{i}.png"
            generate_style_plot(ds_seas.sel(season=s), fname, ranges['seas'][0], ranges['seas'][1])
        except: pass

    # ==========================================
    # 6. EXPORT JSON (+ AREA AVERAGE CALCULATION)
    # ==========================================
    print("\nFinalisasi JSON (Calculating Area Average)...")
    
    print("   ...Calculating Domain Averages")
    aa_ts_raw = ds_combined.mean(dim=['latitude', 'longitude'], skipna=True)
    aa_clim_raw = ds_clim.mean(dim=['latitude', 'longitude'], skipna=True)
    aa_seas_raw = ds_seas.mean(dim=['latitude', 'longitude'], skipna=True)

    # Konversi ke list python
    aa_ts_vals = [round(float(x), 2) if not np.isnan(x) else None for x in aa_ts_raw.values]
    aa_clim_vals = [round(float(x), 2) for x in aa_clim_raw.values]
    aa_seas_vals = []
    for s in season_order:
        try: aa_seas_vals.append(round(float(aa_seas_raw.sel(season=s).values), 2))
        except: aa_seas_vals.append(None)

    # Helper Grid Smooth
    def smooth(da): 
        return da.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()

    ts_smooth = smooth(ds_combined)
    clim_smooth = smooth(ds_clim)
    seas_smooth = smooth(ds_seas)
    
    lats = ts_smooth.latitude.values
    lons = ts_smooth.longitude.values
    labels_clim = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    output_data = {
        "metadata": {
            "latitudes": lats.tolist(),
            "longitudes": lons.tolist(),
            "labels_ts": labels_ts,
            "labels_clim": labels_clim,
            "labels_seas": season_order,
            "spatial_paths": {
                # [MODIFIKASI 5] Path diupdate agar frontend membaca folder frames_mwp
                "ts": f"{base_frame_dir}/ts/spatial_ts_",
                "clim": f"{base_frame_dir}/clim/spatial_clim_",
                "seas": f"{base_frame_dir}/seas/spatial_seas_"
            },
            "colorbar_paths": {
                "ts": f"{base_frame_dir}/ts/colorbar_legend.png",
                "clim": f"{base_frame_dir}/clim/colorbar_legend.png",
                "seas": f"{base_frame_dir}/seas/colorbar_legend.png"
            },
            "bounds": [[float(lats.min()), float(lons.min())], [float(lats.max()), float(lons.max())]],
            "color_ranges": ranges 
        },
        "area_average": {
            "ts": aa_ts_vals,
            "clim": aa_clim_vals,
            "seas": aa_seas_vals
        },
        "grid_data": {}
    }

    # Isi Data Grid
    print("   ...Populating Grid Data")
    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            grid_key = f"{i},{j}"
            
            ts_vals = [round(float(x), 2) if not np.isnan(x) else None for x in ts_smooth[:, i, j].values]
            if all(x is None for x in ts_vals): continue

            clim_vals = [round(float(x), 2) for x in clim_smooth[:, i, j].values]
            seas_vals = []
            for s in season_order:
                try: seas_vals.append(round(float(seas_smooth.sel(season=s)[i, j]), 2))
                except: seas_vals.append(None)

            output_data["grid_data"][grid_key] = {"ts": ts_vals, "clim": clim_vals, "seas": seas_vals}

    # [MODIFIKASI 6] Output ke JSON berbeda agar aman
    json_filename = 'wave_data_mwp.json'
    with open(json_filename, 'w') as f:
        json.dump(output_data, f)
    
    print(f"\nSELESAI! Data MWP tersimpan di '{json_filename}' dan folder '{base_frame_dir}/'")

if __name__ == "__main__":
    process_mwp_high_res_with_custom_cbar()

GENERATOR SPATIAL PLOT: MEAN WAVE PERIOD (MWP)
Membaca data MWP 1979-2024...


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


...

C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\xarray\groupers.py:392: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


   [2024] OK
Menggabungkan dataset...

Processing Time Series...
   550 frames...
Processing Monthly Clim...

Processing Seasonal...

Finalisasi JSON (Calculating Area Average)...
   ...Calculating Domain Averages
   ...Populating Grid Data

SELESAI! Data MWP tersimpan di 'wave_data_mwp.json' dan folder 'frames_mwp/'
